### 3.6.2. Representatividade temporal

<div style="text-align: justify"> Esta subseção apresenta a avaliação da representatividade temporal dos dados medidos pelas estações da rede de monitoramento da qualidade do ar no Brasil. O objetivo é verificar a completude das séries temporais e identificar o grau de representatividade dos dados em diferentes escalas de tempo (horária, diária, mensal e anual), conforme as recomendações do Guia Técnico de Monitoramento e Avaliação da Qualidade do Ar do Ministério do Meio Ambiente <cite id="rb5hg"><a href="#zotero%7C22267313%2FM8BLF2PV">(BRASIL, 2020)</a></cite>.

</p> A representatividade temporal foi calculada a partir da proporção de dados válidos em cada intervalo de agregação, considerando critérios mínimos de completude definidos para cada nível temporal. Esses critérios asseguram que os dados utilizados nas análises apresentem cobertura suficiente ao longo do tempo. Os critérios adotados foram os seguintes:  <br/><br/>
<ul>
        <b><li>Média horária:</b> pelo menos 3/4 das medições válidas dentro da hora;
        <b><li>Média diária:</b> pelo menos 2/3 das médias horárias válidas no dia; 
        <b><li>Média mensal:</b> pelo menos 2/3 das médias diárias válidas no mês; 
        <b><li>Média anual:</b> pelo menos 1/2 das médias diárias válidas obtidas em cada quadrimestre (jan–abr; mai–ago; set–dez). 
    <ul></div>


<div style="text-align: justify"> A Figura 29 apresenta os percentuais de dados válidos por estação e poluente, permitindo a visualização da completude temporal das séries em diferentes escalas.</div>

```{tip}
Na Figura 29, os anos destacados em verde representam períodos com dados considerados representativos, indicando maior consistência e disponibilidade de informações para análise.
```

In [2]:
from pathlib import Path
from IPython.display import HTML

root = Path("../_static/representatividade/rep_temporal")

# Carrega os arquivos gerados (certifique-se de ter rodado o passo anterior de geração dos JSONs)
gj_diario = (root / "rep_temporal_diario.geojson").read_text(encoding="utf-8")
gj_mensal = (root / "rep_temporal_mensal.geojson").read_text(encoding="utf-8")
gj_anual  = (root / "rep_temporal_anual.geojson").read_text(encoding="utf-8")

html_code = """
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8"/>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<style>
  #map {
      width:100%;
      height:640px;
      border:1px solid #ddd;
      position: relative; 
  }
  #selector {
      margin: 10px 0;
      font-size: 14px;
      padding: 4px 6px;
  }
  .bar-container {
      display:flex;
      align-items:center;
      gap:6px;
      margin:2px 0;
  }
  .bar-bg {
      width:120px;
      height:10px;
      border:1px solid #aaa;
  }
  .bar-fill {
      height:100%;
  }
  .legend {
      position: absolute;
      bottom: 20px;
      right: 20px;
      background: white;
      padding: 10px;
      border-radius: 6px;
      font-size: 12px;
      line-height: 1.4;
      box-shadow: 0 1px 4px rgba(0,0,0,0.3);
      z-index: 9999;
  }
  .legend-title {
      font-weight: bold;
      margin-bottom: 6px;
  }
  .legend-item {
      display: flex;
      align-items: center;
      margin: 3px 0;
  }
  .legend-color {
      width: 14px;
      height: 14px;
      margin-right: 6px;
      display: inline-block;
      border: 1px solid #333;
  }
</style>
</head>

<body>

<select id="selector">
  <option value="diario">Representatividade diária</option>
  <option value="mensal">Representatividade mensal</option>
  <option value="anual">Representatividade anual</option>
</select>

<div id="map">
  <div class="legend" id="legend">
    <div class="legend-title">Representatividade (%)</div>
    <div class="legend-item"><span class="legend-color" style="background:#666;"></span>0%</div>
    <div class="legend-item"><span class="legend-color" style="background:#d73027;"></span>1–24%</div>
    <div class="legend-item"><span class="legend-color" style="background:#fc8d59;"></span>25–49%</div>
    <div class="legend-item"><span class="legend-color" style="background:#fee08b;"></span>50–74%</div>
    <div class="legend-item"><span class="legend-color" style="background:#d9ef8b;"></span>75–89%</div>
    <div class="legend-item"><span class="legend-color" style="background:#1a9850;"></span>90–100%</div>
  </div>
</div>

<script id="gj-diario" type="application/json">
""" + gj_diario + """
</script>

<script id="gj-mensal" type="application/json">
""" + gj_mensal + """
</script>

<script id="gj-anual" type="application/json">
""" + gj_anual + """
</script>

<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>

<script>

const map = L.map("map", {
    minZoom: 3.5,
    maxZoom: 13,
    maxBounds: L.latLngBounds([-34,-74],[6,-34]),
    maxBoundsViscosity: 0.8
}).setView([-14.2,-51.9], 4.5);

L.tileLayer(
    "https://cartodb-basemaps-a.global.ssl.fastly.net/light_all/{z}/{x}/{y}.png",
    { attribution: "© OpenStreetMap, © CartoDB" }
).addTo(map);

function cor(v){
    if (v == null || isNaN(v)) return "#ffffff";
    if (v === 0) return "#666666";
    if (v < 25) return "#d73027";
    if (v < 50) return "#fc8d59";
    if (v < 75) return "#fee08b";
    if (v < 90) return "#d9ef8b";
    return "#1a9850";
}

function barra(val, col){
    if (val == null || isNaN(val)) return "—";
    return `
      <div class="bar-container">
        <div class="bar-bg">
          <div class="bar-fill" style="width:${val}%; background:${col};"></div>
        </div>
        <span>${val.toFixed(1)}%</span>
      </div>`;
}

// --- FUNÇÃO ATUALIZADA: Cor Verde Escuro ---
function formatarAnos(monitorados, representativos) {
    if (!monitorados || monitorados.length === 0) return "—";
    
    const setRep = new Set(representativos || []);

    return monitorados.map(ano => {
        if (setRep.has(ano)) {
            // Alterado de <b> para <span> com style color: #006400 (Verde Escuro)
            // Mantive o negrito para legibilidade, se quiser tirar remova 'font-weight: bold;'
            return `<span style="color: #006400; font-weight: bold;">${ano}</span>`;
        }
        return ano;
    }).join(", ");
}

function popupHTML(p, campo){
    const v = p[campo];
    const anosFormatados = formatarAnos(p.ANOS_MONITORADOS, p.ANOS_REPRESENTATIVOS);

    return `
      <div style="font-size:13px;">
        <b>${p.CIDADE} (${p.UF})</b><br>
        <b>ID_MMA:</b> ${p.ID_MMA_COMPLETO}<br>
        <b>ID_OEMA:</b> ${p.ID_OEMA}<br>
        <div style="margin: 4px 0; line-height:1.4;">
            <b>Anos monitorados:</b><br>
            <span style="color:#555;">${anosFormatados}</span>
        </div>
        <hr style="margin:4px 0;">
        ${barra(v, cor(v))}
      </div>`;
}

const dataDiario = JSON.parse(document.getElementById("gj-diario").textContent);
const dataMensal = JSON.parse(document.getElementById("gj-mensal").textContent);
const dataAnual  = JSON.parse(document.getElementById("gj-anual").textContent);

let layerAtual = null;

function carregar(dataset, campo){
    if (layerAtual) map.removeLayer(layerAtual);

    layerAtual = L.geoJSON(dataset, {
        pointToLayer: (f, latlng) => {
            const p = f.properties;
            const valor = p[campo];
            const c = cor(valor);

            const mk = L.circleMarker(latlng, {
                radius:6,
                color:c,
                weight:2,
                fillColor:c,
                fillOpacity:0.9
            });

            mk.bindPopup(popupHTML(p, campo));
            mk.bindTooltip(`${p.CIDADE} (${p.UF})`);
            return mk;
        }
    }).addTo(map);
}

// inicial
carregar(dataDiario, "PRCNT_REP_TEMPORAL_DIARIA");

document.getElementById("selector").addEventListener("change", (e) => {
    if (e.target.value === "diario") carregar(dataDiario, "PRCNT_REP_TEMPORAL_DIARIA");
    if (e.target.value === "mensal") carregar(dataMensal, "PRCNT_REP_TEMPORAL_MENSAL");
    if (e.target.value === "anual")  carregar(dataAnual,  "PRCNT_REP_TEMPORAL_ANUAL");
});
</script>

</body>
</html>
"""

HTML(html_code)

<p style="text-align:center; color:#636D7D; font-size:15.5px;">
  <em>Fig. 29</em> Representatividade temporal das medições nas estações de monitoramento da qualidade do ar.
</p><br/>

### Referências 

<!-- BIBLIOGRAPHY START --> <div class="csl-bib-body"> <div class="csl-entry"><i id="zotero|22267313/M8BLF2PV"></i>BRASIL. <b>Guia Técnico para o Monitoramento e Avaliação da Qualidade do Ar</b>. , 2020. </div> </div>